# Notebook 2 — Accountability Half-Life

**Mandate Vacuum Governance Intelligence**

This notebook models how accountability erodes over time as complaints are transferred between departments.

### Formula
```
R(t) = R₀ · e^(−λt)
t½   = ln(2) / λ
λ    = (transfers / days_open) × scaling_factor
```
- Long half-life → Accountability persists → Resolution likely
- Short half-life → Accountability evaporates → Complaint likely unresolved

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../sample_data/bbmp_complaints_cleaned.csv')
print(f'Loaded {df["complaint_id"].nunique()} unique complaints across {df["category"].nunique()} categories.')
df.head()

In [ ]:
def accountability_halflife(df, complaint_id, scaling_factor=0.5):
    """
    Model accountability decay for a single complaint.
    
    Parameters:
        df             : DataFrame with transfer records
        complaint_id   : Target complaint ID
        scaling_factor : Calibration parameter (default=0.5)
    
    Returns:
        dict with decay metrics and curve data
    """
    complaint = df[df['complaint_id'] == complaint_id].copy()
    n_transfers = len(complaint)
    total_days = complaint['days_open'].max()
    category = complaint['category'].iloc[0]
    resolved = complaint['resolved'].iloc[-1]
    
    if n_transfers < 1 or total_days == 0:
        return None
    
    # Decay constant from transfer rate
    transfer_rate = n_transfers / total_days
    lambda_decay = transfer_rate * scaling_factor
    
    # Half-life in days
    half_life = np.log(2) / lambda_decay if lambda_decay > 0 else float('inf')
    
    # Generate decay curve
    time_points = np.linspace(0, total_days, 200)
    accountability = np.exp(-lambda_decay * time_points)
    final_accountability = float(np.exp(-lambda_decay * total_days))
    
    # Risk classification
    if half_life < 15:
        risk = 'CRITICAL'
        color = 'red'
    elif half_life < 30:
        risk = 'MODERATE'
        color = 'orange'
    else:
        risk = 'STABLE'
        color = 'green'
    
    return {
        'complaint_id': complaint_id,
        'category': category,
        'total_transfers': n_transfers,
        'total_days_open': total_days,
        'resolved': resolved,
        'lambda_decay': round(lambda_decay, 5),
        'half_life_days': round(half_life, 1),
        'final_accountability_pct': round(final_accountability * 100, 1),
        'risk': risk,
        'color': color,
        'time_points': time_points,
        'accountability_curve': accountability
    }

print('Function defined.')

In [ ]:
# Run half-life analysis for all complaints
complaint_ids = df['complaint_id'].unique()
results = [accountability_halflife(df, cid) for cid in complaint_ids]
results = [r for r in results if r is not None]

# Summary table
summary = pd.DataFrame([{
    'complaint_id': r['complaint_id'],
    'category': r['category'],
    'transfers': r['total_transfers'],
    'days_open': r['total_days_open'],
    'half_life_days': r['half_life_days'],
    'final_accountability_%': r['final_accountability_pct'],
    'resolved': r['resolved'],
    'risk': r['risk']
} for r in results])

print('=== ACCOUNTABILITY HALF-LIFE RESULTS ===')
print(summary.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Decay curves for all complaints
ax1 = axes[0]
colors = cm.RdYlGn(np.linspace(0.1, 0.9, len(results)))

for r, col in zip(results, colors):
    ax1.plot(
        r['time_points'],
        r['accountability_curve'],
        label=f"{r['complaint_id']} (t½={r['half_life_days']}d)",
        color=r['color'],
        alpha=0.7,
        linewidth=1.5
    )

ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.6, label='50% Threshold')
ax1.axhline(y=0.25, color='lightgray', linestyle=':', alpha=0.6, label='25% Threshold')
ax1.set_xlabel('Days Since Complaint Filed', fontsize=11)
ax1.set_ylabel('Accountability Remaining', fontsize=11)
ax1.set_title('Accountability Decay Curves\nper Complaint', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=7, loc='upper right')

# Chart 2: Half-life by category (box plot)
ax2 = axes[1]
categories = summary['category'].unique()
data_by_cat = [summary[summary['category'] == cat]['half_life_days'].values for cat in categories]
bp = ax2.boxplot(data_by_cat, labels=categories, patch_artist=True)
cat_colors = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9', '#8e44ad']
for patch, color in zip(bp['boxes'], cat_colors[:len(categories)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.axhline(y=15, color='red', linestyle='--', alpha=0.6, label='Critical threshold (15d)')
ax2.axhline(y=30, color='orange', linestyle='--', alpha=0.6, label='Moderate threshold (30d)')
ax2.set_xlabel('Complaint Category', fontsize=11)
ax2.set_ylabel('Half-Life (Days)', fontsize=11)
ax2.set_title('Half-Life Distribution\nby Complaint Category', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/02_accountability_halflife.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/02_accountability_halflife.png')

In [ ]:
# Key insights
print('=== KEY FINDINGS ===')
critical = summary[summary['risk'] == 'CRITICAL']
stable = summary[summary['risk'] == 'STABLE']

print(f'🔴 CRITICAL (t½ < 15 days): {len(critical)} complaints — accountability evaporates before resolution')
print(f'🟠 MODERATE (15–30 days):   {len(summary[summary["risk"]=="MODERATE"])} complaints')
print(f'🟢 STABLE   (t½ > 30 days): {len(stable)} complaints — accountability persists')

print(f'\nAverage half-life by category:')
print(summary.groupby('category')['half_life_days'].mean().round(1).to_string())

print(f'\nResolution rate by risk level:')
print(summary.groupby('risk')['resolved'].mean().round(3).mul(100).astype(str).add('%').to_string())